[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/18.%20The%20Yard%20Crane%20%28RTG_RMG%29%20Scheduling%20Problem/P18-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 18. The Yard Crane (RTG/RMG) Scheduling Problem
## Tier 3: Grasshopper Optimization Algorithm

### Purpose and scope

This notebook uses the Grasshopper Optimization Algorithm (GOA) to search for a schedule for a set of yard-crane jobs. A schedule assigns every job to one crane and determines the order in which that crane performs its assigned jobs. The example contains 15 jobs and 4 cranes. The algorithm returns a best-found schedule, its fitness, and histories that describe how the population changes during the search.

The method is a stochastic, population-based metaheuristic: it searches for good solutions but does not certify that a globally optimal solution has been found. The implementation uses continuous vectors as a search representation and decodes them into discrete job assignments and sequences. The social interaction and attraction-to-target terms below describe the GOA variant implemented in this notebook; no wind-advection term is used.

### Modeling assumptions

- Each job is assigned to exactly one crane, and each crane processes its assigned jobs sequentially.
- A crane starts at its specified bay position at time zero. Travel time is the absolute difference between bay positions, in minutes per bay unit.
- A job cannot start before its release time. A crane may wait at the job location until the job is released.
- Processing times, release times, due dates, and priority weights are known and deterministic.
- Cranes do not block one another, share no capacity constraints, and have no setup times. A job's location is treated as its destination; the operational details of storage, retrieval, and relocation are not modeled separately.
- The fitness combines normalized makespan, weighted tardiness, and travel. The three terms have fixed weights and reference scales specified in the code.

### Scheduling model

For crane \(k\), let \(\pi_k=(j_{k,1},\ldots,j_{k,n_k})\) be its ordered job sequence. Let \(p_j\), \(r_j\), \(d_j\), \(w_j\), and \(\ell_j\) denote job \(j\)'s processing time, release time, due date, priority weight, and location. Let \(q_k\) be crane \(k\)'s initial position. Set \(C_{k,0}=0\) and \(L_{k,0}=q_k\). For each position \(m=1,\ldots,n_k\), the travel time, start time, and completion time are

$$\
\tau_{k,m}=|L_{k,m-1}-\ell_{j_{k,m}}|,\qquad
S_{k,m}=\max\{C_{k,m-1}+\tau_{k,m},\ r_{j_{k,m}}\},\qquad
C_{k,m}=S_{k,m}+p_{j_{k,m}},\qquad L_{k,m}=\ell_{j_{k,m}}.
$$

The completion time \(C_{k,m}\) includes travel, any waiting for release, and processing. Job tardiness is \(T_j=\max\{0,C_j-d_j\}\), where \(C_j\) is the completion time of the job in its crane sequence. The makespan is \(M=\max_k C_{k,n_k}\), with an empty crane contributing zero. Total weighted tardiness is \(D=\sum_j w_jT_j\), and total travel is \(R=\sum_{k,m}\tau_{k,m}\). The minimization objective used by the code is

$$\
F=0.40\frac{M}{B_M}+0.35\frac{D}{B_D}+0.25\frac{R}{B_R}.
$$

Here \(B_M\), \(B_D\), and \(B_R\) are reference scales calculated in `evaluate_fitness`; they normalize the terms but are not claimed to be tight mathematical upper bounds. Lower fitness is preferred. The assignment and sequence together must include each job exactly once. Several distinct schedules can have the same fitness, so a best-found solution need not be unique.

### GOA search representation and update

A grasshopper is one candidate schedule, represented by a real-valued vector \(X_i\in[0,1]^D\). Its first \(n\) coordinates encode crane assignments for \(n\) jobs; the remaining \(Kn\) coordinates encode job-order priorities for \(K\) cranes. Decoding rounds the assignment coordinates to crane indices and sorts each crane's assigned jobs by its corresponding priorities. This mapping lets a continuous swarm search act on a discrete scheduling problem.

In each iteration, grasshopper \(i\) is affected by social interactions with the other population members and by attraction toward the best position found so far. In this implementation, distances are divided by \(\sqrt{D}\) before applying the social interaction function. With \(N\) grasshoppers, dimension \(D\), target position \(T\), and decreasing coefficient \(c\), the update is

$$\
X_i\leftarrow\operatorname{clip}_{[0,1]^D}\!\left[
X_i+c\sum_{j\ne i}s(\rho_{ij})\,\hat{u}_{ij}+c(T-X_i)
\right],\qquad
\rho_{ij}=\frac{\lVert X_j-X_i\rVert_2}{\sqrt{D}},\qquad
\hat{u}_{ij}=\frac{X_j-X_i}{\lVert X_j-X_i\rVert_2}.
$$

When two positions coincide, their social contribution is defined as zero. The interaction function is

$$\
s(r)=0.5e^{-r/1.5}-e^{-r}.
$$

At short distances \(s(r)\) is negative and contributes repulsion; at larger distances it becomes positive and contributes attraction. The target term draws the candidate toward the best-known position. The coefficient decreases linearly from `c_max` toward `c_min`, reducing the size of updates as the search proceeds. The code updates grasshoppers sequentially within each iteration, so a later grasshopper can interact with positions already updated during that iteration.

The best-known position is decoded and evaluated using the scheduling objective. Each iteration records the best fitness and the mean pairwise Euclidean distance between population positions, called population diversity here. These histories describe search progress; they do not prove convergence to a global optimum.

### Computational effort

For population size \(N\), vector dimension \(D=n+Kn\), and \(I\) iterations, calculating all pairwise social forces takes \(O(IN^2D)\) time. Fitness evaluation decodes assignments and sequences and simulates the jobs, requiring approximately \(O(N(n+K\log n))\) work per iteration for this implementation. Storing the population requires \(O(ND)\) space. The optional random baseline performs a separate fixed number of sampled evaluations; it is included only as a reference comparison.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import time

# Fixed seeds make the stochastic example reproducible within the same software environment.
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class Job:
    """A container-handling job with timing, location, and priority data."""

    id: int
    location: int
    processing_time: float
    release_time: float
    due_date: float
    priority_weight: float
    job_type: str


@dataclass
class Crane:
    """A yard crane and its initial bay position."""

    id: int
    start_position: int
    crane_type: str


@dataclass
class Grasshopper:
    """A candidate schedule and its current and best-known states."""

    position: np.ndarray
    fitness: float
    best_position: np.ndarray
    best_fitness: float


@dataclass
class GOAResult:
    """The best schedule and search diagnostics returned by GOA."""

    best_solution: Dict[int, List[int]]
    best_fitness: float
    convergence_history: List[float]
    diversity_history: List[float]
    computation_time: float

In [ ]:
def create_goa_example() -> Tuple[List[Job], List[Crane]]:
    """Create the fixed 15-job, 4-crane scheduling example."""

    jobs = [
        Job(id=1, location=1, processing_time=4, release_time=0, due_date=20, priority_weight=1.5, job_type="storage"),
        Job(id=2, location=3, processing_time=3, release_time=2, due_date=15, priority_weight=2.0, job_type="retrieval"),
        Job(id=3, location=5, processing_time=5, release_time=1, due_date=25, priority_weight=1.0, job_type="relocation"),
        Job(id=4, location=2, processing_time=2, release_time=3, due_date=12, priority_weight=2.5, job_type="retrieval"),
        Job(id=5, location=8, processing_time=4, release_time=0, due_date=22, priority_weight=1.2, job_type="storage"),
        Job(id=6, location=6, processing_time=3, release_time=4, due_date=18, priority_weight=1.8, job_type="retrieval"),
        Job(id=7, location=9, processing_time=6, release_time=2, due_date=30, priority_weight=1.0, job_type="relocation"),
        Job(id=8, location=4, processing_time=2, release_time=5, due_date=14, priority_weight=2.2, job_type="retrieval"),
        Job(id=9, location=7, processing_time=4, release_time=1, due_date=24, priority_weight=1.3, job_type="storage"),
        Job(id=10, location=10, processing_time=3, release_time=3, due_date=16, priority_weight=1.9, job_type="retrieval"),
        Job(id=11, location=0, processing_time=5, release_time=0, due_date=28, priority_weight=1.1, job_type="storage"),
        Job(id=12, location=12, processing_time=2, release_time=6, due_date=13, priority_weight=2.4, job_type="retrieval"),
        Job(id=13, location=11, processing_time=4, release_time=2, due_date=21, priority_weight=1.6, job_type="storage"),
        Job(id=14, location=13, processing_time=3, release_time=4, due_date=17, priority_weight=2.1, job_type="retrieval"),
        Job(id=15, location=14, processing_time=5, release_time=1, due_date=26, priority_weight=1.4, job_type="relocation"),
    ]

    cranes = [
        Crane(id=1, start_position=0, crane_type="RMG"),
        Crane(id=2, start_position=5, crane_type="RMG"),
        Crane(id=3, start_position=10, crane_type="RMG"),
        Crane(id=4, start_position=15, crane_type="RMG"),
    ]

    return jobs, cranes


jobs, cranes = create_goa_example()

print("=== GOA EXAMPLE SETUP ===")
print(f"Jobs: {len(jobs)}, Cranes: {len(cranes)}")
print("\nJob summary:")
job_types = {}
for job in jobs:
    job_types[job.job_type] = job_types.get(job.job_type, 0) + 1
for job_type, count in job_types.items():
    print(f"  {job_type.title()}: {count} jobs")

print("\nCrane starting positions:")
for crane in cranes:
    print(f"  Crane {crane.id}: bay {crane.start_position}")

In [ ]:
def calculate_travel_time(pos1: int, pos2: int) -> float:
    """Return travel time in minutes for the one-dimensional bay layout."""
    return abs(pos1 - pos2) * 1.0


def encode_solution(
    jobs: List[Job],
    cranes: List[Crane],
    crane_assignments: List[int],
    job_sequences: Dict[int, List[int]],
) -> np.ndarray:
    """Encode assignments and within-crane sequence priorities in [0, 1]."""
    n_jobs = len(jobs)
    n_cranes = len(cranes)
    if len(crane_assignments) != n_jobs:
        raise ValueError("There must be one crane assignment for each job.")
    if n_cranes == 0:
        raise ValueError("At least one crane is required.")

    encoding = np.zeros(n_jobs + n_jobs * n_cranes, dtype=float)

    for job_index, crane_index in enumerate(crane_assignments):
        if not 0 <= crane_index < n_cranes:
            raise ValueError("Crane assignment indices must be valid crane indices.")
        encoding[job_index] = (
            crane_index / (n_cranes - 1) if n_cranes > 1 else 0.5
        )

    offset = n_jobs
    for crane_index in range(n_cranes):
        sequence = job_sequences.get(crane_index, [])
        for rank, job_id in enumerate(sequence):
            if not 1 <= job_id <= n_jobs:
                raise ValueError("Job IDs must be consecutive integers from 1 to n.")
            encoding[offset + crane_index * n_jobs + (job_id - 1)] = (
                rank / len(sequence) if sequence else 0.0
            )

    return encoding


def decode_solution(
    encoding: np.ndarray,
    jobs: List[Job],
    cranes: List[Crane],
) -> Dict[int, List[int]]:
    """Decode a continuous vector into crane assignments and job sequences."""
    n_jobs = len(jobs)
    n_cranes = len(cranes)
    expected_size = n_jobs + n_jobs * n_cranes
    if encoding.size != expected_size:
        raise ValueError(f"Expected an encoding of length {expected_size}.")
    if n_cranes == 0:
        raise ValueError("At least one crane is required.")

    bounded_encoding = np.clip(encoding, 0.0, 1.0)
    if n_cranes > 1:
        assignment_values = np.rint(
            bounded_encoding[:n_jobs] * (n_cranes - 1)
        ).astype(int)
    else:
        assignment_values = np.zeros(n_jobs, dtype=int)

    crane_jobs = {crane_index: [] for crane_index in range(n_cranes)}
    for job_index, crane_index in enumerate(assignment_values):
        crane_jobs[int(crane_index)].append(job_index + 1)

    solution = {}
    offset = n_jobs
    for crane_index, job_list in crane_jobs.items():
        priorities = bounded_encoding[
            offset + crane_index * n_jobs : offset + (crane_index + 1) * n_jobs
        ]
        solution[crane_index] = sorted(
            job_list,
            key=lambda job_id: (priorities[job_id - 1], job_id),
        )

    return solution

In [ ]:
def evaluate_fitness(
    solution: Dict[int, List[int]],
    jobs: List[Job],
    cranes: List[Crane],
) -> float:
    """Evaluate a complete schedule; lower weighted fitness is better."""
    expected_cranes = set(range(len(cranes)))
    if set(solution) != expected_cranes:
        raise ValueError("The solution must contain one sequence for every crane.")

    scheduled_jobs = [job_id for sequence in solution.values() for job_id in sequence]
    expected_jobs = [job.id for job in jobs]
    if len(scheduled_jobs) != len(set(scheduled_jobs)) or set(scheduled_jobs) != set(expected_jobs):
        raise ValueError("Every job must appear exactly once in the schedule.")

    crane_completion_times = {}
    total_weighted_delay = 0.0
    total_travel = 0.0

    for crane_index, job_sequence in solution.items():
        current_position = cranes[crane_index].start_position
        current_time = 0.0

        for job_id in job_sequence:
            job = jobs[job_id - 1]
            travel_time = calculate_travel_time(current_position, job.location)
            total_travel += travel_time
            arrival_time = current_time + travel_time
            start_time = max(arrival_time, job.release_time)
            completion_time = start_time + job.processing_time
            delay = max(0.0, completion_time - job.due_date)
            total_weighted_delay += delay * job.priority_weight

            current_position = job.location
            current_time = completion_time

        crane_completion_times[crane_index] = current_time

    makespan = max(crane_completion_times.values(), default=0.0)
    # These fixed reference scales normalize objective components; they are not tight bounds.
    makespan_scale = sum(job.processing_time for job in jobs) + 100.0
    delay_scale = max(
        sum(
            (job.due_date - job.release_time) * job.priority_weight
            for job in jobs
        ),
        1.0,
    )
    travel_scale = max(sum(job.processing_time for job in jobs) * 2.0, 1.0)

    normalized_makespan = makespan / makespan_scale
    normalized_delay = total_weighted_delay / delay_scale
    normalized_travel = total_travel / travel_scale

    return (
        0.40 * normalized_makespan
        + 0.35 * normalized_delay
        + 0.25 * normalized_travel
    )


def initialize_population(
    jobs: List[Job],
    cranes: List[Crane],
    population_size: int,
) -> List[Grasshopper]:
    """Create random candidate schedules and their initial fitness values."""
    if population_size < 1:
        raise ValueError("Population size must be at least one.")
    if not cranes:
        raise ValueError("At least one crane is required.")

    n_jobs = len(jobs)
    n_cranes = len(cranes)
    population = []

    for _ in range(population_size):
        crane_assignments = [
            random.randint(0, n_cranes - 1) for _ in range(n_jobs)
        ]
        job_sequences = {}
        for crane_index in range(n_cranes):
            assigned_jobs = [
                job_index + 1
                for job_index, assigned_crane in enumerate(crane_assignments)
                if assigned_crane == crane_index
            ]
            random.shuffle(assigned_jobs)
            job_sequences[crane_index] = assigned_jobs

        position = encode_solution(jobs, cranes, crane_assignments, job_sequences)
        solution = decode_solution(position, jobs, cranes)
        fitness = evaluate_fitness(solution, jobs, cranes)
        population.append(
            Grasshopper(
                position=position,
                fitness=fitness,
                best_position=position.copy(),
                best_fitness=fitness,
            )
        )

    return population

In [ ]:
def social_interaction_function(distance: float) -> float:
    """Evaluate the GOA attraction-repulsion function at a scaled distance."""
    return 0.5 * np.exp(-distance / 1.5) - np.exp(-distance)


def calculate_social_forces(
    population: List[Grasshopper],
    current_idx: int,
    c: float,
) -> np.ndarray:
    """Sum scaled social interactions from all other grasshoppers."""
    current = population[current_idx]
    social_force = np.zeros_like(current.position)
    dimension_scale = np.sqrt(current.position.size)

    for other_idx, other in enumerate(population):
        if other_idx == current_idx:
            continue

        difference = other.position - current.position
        raw_distance = np.linalg.norm(difference)
        if raw_distance > 0.0:
            scaled_distance = raw_distance / dimension_scale
            unit_direction = difference / raw_distance
            social_force += c * social_interaction_function(scaled_distance) * unit_direction

    return social_force


def calculate_gravity_force(
    current: Grasshopper,
    best_global: np.ndarray,
    c: float,
) -> np.ndarray:
    """Calculate the coefficient-scaled attraction toward the best-known position."""
    return c * (best_global - current.position)


def update_position(
    current: Grasshopper,
    social_force: np.ndarray,
    gravity_force: np.ndarray,
    best_global: np.ndarray,
    jobs: List[Job],
    cranes: List[Crane],
    bounds: Tuple[float, float],
) -> np.ndarray:
    """Apply social and target forces, then keep the encoding within its bounds."""
    new_position = current.position + social_force + gravity_force
    return np.clip(new_position, bounds[0], bounds[1])


def calculate_diversity(population: List[Grasshopper]) -> float:
    """Return the mean pairwise Euclidean distance in the population."""
    if len(population) <= 1:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for first_index in range(len(population)):
        for second_index in range(first_index + 1, len(population)):
            total_distance += np.linalg.norm(
                population[first_index].position
                - population[second_index].position
            )
            pair_count += 1

    return total_distance / pair_count

In [ ]:
def grasshopper_optimization_algorithm(
    jobs: List[Job],
    cranes: List[Crane],
    max_iterations: int = 150,
    population_size: int = 30,
    c_max: float = 1.0,
    c_min: float = 0.00001,
) -> GOAResult:
    """Search for a low-fitness crane schedule using the GOA population."""
    if max_iterations < 1:
        raise ValueError("Maximum iterations must be at least one.")
    if c_max < c_min or c_min < 0.0:
        raise ValueError("Coefficients must satisfy c_max >= c_min >= 0.")

    print("=== GRASSHOPPER OPTIMIZATION ALGORITHM ===")
    print(f"Population size: {population_size}, Maximum iterations: {max_iterations}")
    print(f"Jobs: {len(jobs)}, Cranes: {len(cranes)}\n")
    start_time = time.time()

    population = initialize_population(jobs, cranes, population_size)
    best_grasshopper = min(population, key=lambda grasshopper: grasshopper.fitness)
    best_global_position = best_grasshopper.position.copy()
    best_global_fitness = best_grasshopper.fitness

    convergence_history = [best_global_fitness]
    diversity_history = [calculate_diversity(population)]
    print(f"Initial best fitness: {best_global_fitness:.4f}")
    print(f"Initial diversity: {diversity_history[0]:.4f}\n")

    completed_iterations = 0
    for iteration in range(max_iterations):
        c = c_max - iteration * (c_max - c_min) / max_iterations

        for grasshopper_index, grasshopper in enumerate(population):
            social_force = calculate_social_forces(
                population, grasshopper_index, c
            )
            gravity_force = calculate_gravity_force(
                grasshopper, best_global_position, c
            )
            new_position = update_position(
                grasshopper,
                social_force,
                gravity_force,
                best_global_position,
                jobs,
                cranes,
                (0.0, 1.0),
            )

            new_solution = decode_solution(new_position, jobs, cranes)
            new_fitness = evaluate_fitness(new_solution, jobs, cranes)
            grasshopper.position = new_position
            grasshopper.fitness = new_fitness

            if new_fitness < grasshopper.best_fitness:
                grasshopper.best_fitness = new_fitness
                grasshopper.best_position = new_position.copy()

            if new_fitness < best_global_fitness:
                best_global_fitness = new_fitness
                best_global_position = new_position.copy()

        completed_iterations = iteration + 1
        convergence_history.append(best_global_fitness)
        diversity_history.append(calculate_diversity(population))

        if completed_iterations % 25 == 0:
            print(
                f"Iteration {completed_iterations:3d}: "
                f"Best fitness = {best_global_fitness:.4f}, "
                f"c = {c:.4f}, Diversity = {diversity_history[-1]:.4f}"
            )

        if completed_iterations > 50:
            recent_improvement = abs(
                convergence_history[-1] - convergence_history[-20]
            )
            if recent_improvement < 0.0001:
                print(
                    f"\nEarly stopping at iteration {completed_iterations} "
                    "because recent best fitness has changed very little."
                )
                break

    final_solution = decode_solution(best_global_position, jobs, cranes)
    computation_time = time.time() - start_time
    print("\n=== OPTIMIZATION COMPLETE ===")
    print(f"Final best fitness: {best_global_fitness:.4f}")
    print(f"Computation time: {computation_time:.2f} seconds")
    print(f"Iterations completed: {completed_iterations}")

    return GOAResult(
        best_solution=final_solution,
        best_fitness=best_global_fitness,
        convergence_history=convergence_history,
        diversity_history=diversity_history,
        computation_time=computation_time,
    )

In [ ]:
# The fixed parameters make this run reproducible with the seeds set above.
goa_result = grasshopper_optimization_algorithm(
    jobs=jobs,
    cranes=cranes,
    max_iterations=150,
    population_size=30,
    c_max=1.0,
    c_min=0.00001,
)

In [ ]:
def analyze_goa_solution(
    result: GOAResult,
    jobs: List[Job],
    cranes: List[Crane],
    verbose: bool = True,
) -> Dict:
    """Reconstruct job timings and summarize a schedule's performance."""
    analysis = {
        "crane_assignments": result.best_solution,
        "detailed_schedule": {},
        "performance_metrics": {},
    }
    job_details = {}
    completion_times = {}

    if verbose:
        print("\n=== GOA SOLUTION ANALYSIS ===")

    for crane_index, job_sequence in result.best_solution.items():
        crane = cranes[crane_index]
        if verbose:
            print(f"\nCrane {crane.id} schedule:")

        current_position = crane.start_position
        current_time = 0.0
        crane_schedule = []

        if not job_sequence and verbose:
            print("  No jobs assigned")

        for step_index, job_id in enumerate(job_sequence, start=1):
            job = jobs[job_id - 1]
            travel_time = calculate_travel_time(current_position, job.location)
            arrival_time = current_time + travel_time
            start_time = max(arrival_time, job.release_time)
            completion_time = start_time + job.processing_time
            delay = max(0.0, completion_time - job.due_date)
            weighted_delay = delay * job.priority_weight

            if verbose:
                print(f"  Step {step_index}: Job {job_id} ({job.job_type})")
                print(
                    f"    Bay: {job.location}, Travel: {travel_time:.1f} minutes, "
                    f"Start: {start_time:.1f}, Completion: {completion_time:.1f}"
                )
                print(
                    f"    Tardiness: {delay:.1f} minutes, "
                    f"Weighted tardiness: {weighted_delay:.2f}"
                )

            job_details[job_id] = {
                "crane_index": crane_index,
                "sequence_position": step_index,
                "arrival_time": arrival_time,
                "start_time": start_time,
                "completion_time": completion_time,
                "travel_time": travel_time,
                "delay": delay,
                "weighted_delay": weighted_delay,
            }
            crane_schedule.append(
                {
                    "job_id": job_id,
                    "arrival_time": arrival_time,
                    "start_time": start_time,
                    "completion_time": completion_time,
                    "location": job.location,
                }
            )
            current_position = job.location
            current_time = completion_time

        completion_times[crane_index] = current_time
        analysis["detailed_schedule"][crane_index] = crane_schedule

    makespan = max(completion_times.values(), default=0.0)
    total_weighted_delay = sum(
        details["weighted_delay"] for details in job_details.values()
    )
    crane_utilization = {}
    for crane_index, job_sequence in result.best_solution.items():
        processing_time = sum(
            jobs[job_id - 1].processing_time for job_id in job_sequence
        )
        crane_utilization[crane_index] = (
            100.0 * processing_time / makespan if makespan > 0.0 else 0.0
        )

    analysis["performance_metrics"] = {
        "makespan": makespan,
        "total_weighted_delay": total_weighted_delay,
        "crane_utilization": crane_utilization,
        "total_jobs": len(jobs),
        "assigned_jobs": sum(
            len(sequence) for sequence in result.best_solution.values()
        ),
    }

    if verbose:
        print("\n=== PERFORMANCE SUMMARY ===")
        print(f"Makespan: {makespan:.1f} minutes")
        print(f"Total weighted tardiness: {total_weighted_delay:.2f}")
        print(
            f"Jobs assigned: {analysis['performance_metrics']['assigned_jobs']}"
            f"/{len(jobs)}"
        )
        print("\nCrane utilization (processing time / makespan):")
        for crane_index, utilization in crane_utilization.items():
            crane_id = cranes[crane_index].id
            job_count = len(result.best_solution[crane_index])
            print(
                f"  Crane {crane_id}: {utilization:.1f}% "
                f"({job_count} jobs)"
            )

    return analysis


goa_analysis = analyze_goa_solution(goa_result, jobs, cranes)

# These assertions connect the reported schedule to the objective evaluation.
assert goa_analysis["performance_metrics"]["assigned_jobs"] == len(jobs)
assert np.isclose(
    evaluate_fitness(goa_result.best_solution, jobs, cranes),
    goa_result.best_fitness,
    rtol=1e-10,
    atol=1e-12,
)

In [ ]:
def visualize_goa_results(
    result: GOAResult,
    analysis: Dict,
    jobs: List[Job],
    cranes: List[Crane],
) -> None:
    """Plot search histories, processing intervals, and schedule metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Grasshopper Optimization Algorithm: Yard Crane Scheduling",
        fontsize=16,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    iterations = range(len(result.convergence_history))
    ax1.plot(iterations, result.convergence_history, color="tab:blue", linewidth=2)
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Best fitness (lower is better)")
    ax1.set_title("Best-Fitness History")
    ax1.grid(True, alpha=0.3)
    initial_fitness = result.convergence_history[0]
    final_fitness = result.convergence_history[-1]
    if initial_fitness > 0.0:
        improvement = 100.0 * (initial_fitness - final_fitness) / initial_fitness
        ax1.annotate(
            f"Change from initial best: {improvement:.1f}%",
            xy=(len(result.convergence_history) - 1, final_fitness),
            xytext=(0.04, 0.95),
            textcoords="axes fraction",
            arrowprops={"arrowstyle": "->", "color": "tab:red"},
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8},
        )

    ax2 = axes[0, 1]
    ax2.plot(
        range(len(result.diversity_history)),
        result.diversity_history,
        color="tab:green",
        linewidth=2,
    )
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Mean pairwise Euclidean distance")
    ax2.set_title("Population Diversity")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    job_types = list(dict.fromkeys(job.job_type for job in jobs))
    palette = plt.get_cmap("Set2")
    type_colors = {
        job_type: palette(index % palette.N)
        for index, job_type in enumerate(job_types)
    }
    for crane_index, job_sequence in result.best_solution.items():
        crane_id = cranes[crane_index].id
        for job_id in job_sequence:
            job = jobs[job_id - 1]
            job_info = next(
                item
                for item in analysis["detailed_schedule"][crane_index]
                if item["job_id"] == job_id
            )
            start = job_info["start_time"]
            duration = job.processing_time
            ax3.barh(
                crane_id,
                duration,
                left=start,
                height=0.6,
                color=type_colors[job.job_type],
                alpha=0.85,
            )
            ax3.text(
                start + duration / 2.0,
                crane_id,
                f"J{job_id}",
                ha="center",
                va="center",
                color="black",
                fontsize=8,
            )
    ax3.set_xlabel("Time (minutes)")
    ax3.set_ylabel("Crane ID")
    ax3.set_title("Scheduled Processing Intervals")
    ax3.grid(True, alpha=0.3, axis="x")
    ax3.set_yticks([crane.id for crane in cranes])
    makespan = analysis["performance_metrics"]["makespan"]
    ax3.set_xlim(0.0, max(makespan + 5.0, 1.0))
    ax3.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color=type_colors[job_type], label=job_type.title())
            for job_type in job_types
        ],
        title="Job type",
    )
    ax3.text(
        0.01,
        -0.22,
        "Bars show processing intervals; travel and release-time waiting are not drawn.",
        transform=ax3.transAxes,
        fontsize=8,
    )

    ax4 = axes[1, 1]
    metrics = analysis["performance_metrics"]
    metric_names = ["Makespan (minutes)", "Weighted tardiness"]
    metric_values = [metrics["makespan"], metrics["total_weighted_delay"]]
    bars = ax4.bar(
        metric_names,
        metric_values,
        color=["tab:blue", "tab:orange"],
        alpha=0.85,
    )
    ax4.set_ylabel("Value")
    ax4.set_title("Schedule Performance Metrics")
    ax4.grid(True, alpha=0.3, axis="y")
    label_offset = max(metric_values, default=0.0) * 0.01
    for bar, value in zip(bars, metric_values):
        ax4.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + label_offset,
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    utilization_text = "Crane utilization:\n"
    for crane_index, utilization in metrics["crane_utilization"].items():
        utilization_text += f"Crane {cranes[crane_index].id}: {utilization:.1f}%\n"
    ax4.text(
        0.02,
        0.98,
        utilization_text,
        transform=ax4.transAxes,
        fontsize=9,
        va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )
    ax4.text(
        0.98,
        0.98,
        f"Elapsed time: {result.computation_time:.2f} s\n"
        f"Iterations recorded: {len(result.convergence_history) - 1}\n"
        f"Best fitness: {result.best_fitness:.4f}",
        transform=ax4.transAxes,
        fontsize=9,
        va="top",
        ha="right",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


visualize_goa_results(goa_result, goa_analysis, jobs, cranes)

In [ ]:
def compare_with_baselines(
    result: GOAResult,
    analysis: Dict,
    jobs: List[Job],
    cranes: List[Crane],
) -> pd.DataFrame:
    """Compare GOA with the included random-sampling and round-robin references."""
    def random_baseline(
        baseline_jobs: List[Job],
        baseline_cranes: List[Crane],
        trials: int = 100,
    ) -> Dict:
        """Return the best schedule among a fixed number of random samples."""
        best_fitness = float("inf")
        best_solution = None
        for _ in range(trials):
            assignments = [
                random.randint(0, len(baseline_cranes) - 1)
                for _ in range(len(baseline_jobs))
            ]
            solution = {index: [] for index in range(len(baseline_cranes))}
            for job_index, crane_index in enumerate(assignments):
                solution[crane_index].append(job_index + 1)
            for sequence in solution.values():
                random.shuffle(sequence)

            fitness = evaluate_fitness(solution, baseline_jobs, baseline_cranes)
            if fitness < best_fitness:
                best_fitness = fitness
                best_solution = solution

        return {"solution": best_solution, "fitness": best_fitness}

    def greedy_baseline(
        baseline_jobs: List[Job],
        baseline_cranes: List[Crane],
    ) -> Dict:
        """Assign due-date-ordered jobs to cranes in round-robin order."""
        sorted_jobs = sorted(
            baseline_jobs,
            key=lambda job: (job.due_date, -job.priority_weight),
        )
        solution = {index: [] for index in range(len(baseline_cranes))}
        for job_index, job in enumerate(sorted_jobs):
            solution[job_index % len(baseline_cranes)].append(job.id)
        return {
            "solution": solution,
            "fitness": evaluate_fitness(solution, baseline_jobs, baseline_cranes),
        }

    print("=== BASELINE COMPARISON ===")
    random_start = time.time()
    random_result = random_baseline(jobs, cranes, trials=100)
    random_elapsed = time.time() - random_start

    greedy_start = time.time()
    greedy_result = greedy_baseline(jobs, cranes)
    greedy_elapsed = time.time() - greedy_start

    random_analysis = analyze_goa_solution(
        GOAResult(random_result["solution"], random_result["fitness"], [], [], random_elapsed),
        jobs,
        cranes,
        verbose=False,
    )
    greedy_analysis = analyze_goa_solution(
        GOAResult(greedy_result["solution"], greedy_result["fitness"], [], [], greedy_elapsed),
        jobs,
        cranes,
        verbose=False,
    )

    rows = [
        {
            "Algorithm": "GOA",
            "Fitness": result.best_fitness,
            "Makespan (minutes)": analysis["performance_metrics"]["makespan"],
            "Weighted tardiness": analysis["performance_metrics"]["total_weighted_delay"],
            "Elapsed time (seconds)": result.computation_time,
        },
        {
            "Algorithm": "Round-robin due-date reference",
            "Fitness": greedy_result["fitness"],
            "Makespan (minutes)": greedy_analysis["performance_metrics"]["makespan"],
            "Weighted tardiness": greedy_analysis["performance_metrics"]["total_weighted_delay"],
            "Elapsed time (seconds)": greedy_elapsed,
        },
        {
            "Algorithm": "Random sampling (best of 100)",
            "Fitness": random_result["fitness"],
            "Makespan (minutes)": random_analysis["performance_metrics"]["makespan"],
            "Weighted tardiness": random_analysis["performance_metrics"]["total_weighted_delay"],
            "Elapsed time (seconds)": random_elapsed,
        },
    ]
    comparison_df = pd.DataFrame(rows)
    print("\nPerformance comparison:")
    print(comparison_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    reference_fitnesses = [greedy_result["fitness"], random_result["fitness"]]
    if all(fitness > 0.0 for fitness in reference_fitnesses):
        improvements = [
            100.0 * (fitness - result.best_fitness) / fitness
            for fitness in reference_fitnesses
        ]
        print("\nGOA fitness difference relative to each reference:")
        print(f"  Versus round-robin due-date reference: {improvements[0]:.1f}%")
        print(f"  Versus random-sampling reference: {improvements[1]:.1f}%")
    else:
        print("\nRelative percentages are omitted when a reference fitness is zero.")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    labels = [row["Algorithm"] for row in rows]
    fitnesses = [row["Fitness"] for row in rows]
    bars = ax1.bar(labels, fitnesses, color=["tab:green", "tab:orange", "tab:red"])
    ax1.set_ylabel("Fitness (lower is better)")
    ax1.set_title("Fitness Comparison")
    ax1.tick_params(axis="x", rotation=15)
    ax1.grid(True, alpha=0.3, axis="y")
    offset = max(fitnesses, default=0.0) * 0.01
    for bar, value in zip(bars, fitnesses):
        ax1.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + offset,
            f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    elapsed_times = [row["Elapsed time (seconds)"] for row in rows]
    time_bars = ax2.bar(labels, elapsed_times, color=["tab:green", "tab:orange", "tab:red"])
    ax2.set_ylabel("Elapsed time (seconds)")
    ax2.set_title("Measured Runtime for This Execution")
    ax2.tick_params(axis="x", rotation=15)
    ax2.grid(True, alpha=0.3, axis="y")
    time_offset = max(elapsed_times, default=0.0) * 0.01
    for bar, value in zip(time_bars, elapsed_times):
        ax2.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + time_offset,
            f"{value:.3f}",
            ha="center",
            va="bottom",
        )

    plt.suptitle("Schedule Search Reference Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()
    return comparison_df


comparison_df = compare_with_baselines(goa_result, goa_analysis, jobs, cranes)

## Interpreting the search

GOA maintains a population of candidate schedules rather than making a single irreversible greedy choice. Social interaction changes candidate positions in response to other population members, while attraction to the best-known position directs the search toward a promising region. The decreasing coefficient controls the scale of these movements. The continuous update is followed by decoding, so fitness is always evaluated on a discrete schedule rather than on an abstract vector alone.

Use the convergence plot to inspect how the best observed fitness changes; it should be non-increasing because the stored global best is retained. Population diversity measures separation in the continuous encoding, not the number of distinct decoded schedules. Different vectors can decode to the same schedule, and a small diversity value does not establish that the schedule is globally optimal.

The schedule analysis reports travel, release-time waiting implicitly through the gap between arrival and start, processing completion, and weighted tardiness. Crane utilization is defined here as that crane's total processing time divided by the schedule makespan; it is not a measure of travel or waiting. In the Gantt chart, bars show processing intervals only, so travel and waiting are omitted visually even though they affect the schedule times and objective.

The baseline comparison is descriptive, not a proof that GOA is better for every instance. Random sampling is reported as its best result across the specified trials, while the round-robin reference assigns due-date-ordered jobs cyclically to cranes. Runtime depends on the execution environment and should be interpreted as a measurement from the current run, not as a hardware-independent algorithm guarantee.

### Strengths and limitations

A population-based search can explore multiple schedules at once and may find useful trade-offs among makespan, weighted tardiness, and travel. The same flexibility comes with computational cost and sensitivity to population and coefficient settings. This implementation provides no optimality certificate, and its continuous encoding and decoding impose a particular mapping between vectors and discrete schedules.

The example is intentionally limited to independent cranes moving along a one-dimensional bay layout. Real terminals may also require crane interference constraints, safety separation, precedence relations, multiple jobs at a location, equipment availability, setup times, uncertain processing or travel times, and additional operational costs. Those features would require extending the scheduling model and its fitness evaluation while retaining the search framework presented here.